# Human-in-the-Loop: आधीचा क्रियापदाचा दरवाजा, धोका स्तरीकरण, आणि ऑडिट लॉगिंग

या धड्याचा README Human-in-the-Loop ची ओळख करतो एका लहान स्निपेटसह जो वापरकर्त्याला विचारतो `APPROVE` किंवा `REJECT` एजंटने आधीच उत्तर दिल्यानंतर. हा नमुना सुरूवातीस चांगला आहे, पण उत्पादनातील HITL अंमलबजावणीत सामान्यतः आणखी तीन भागांची गरज असते:

1. एक **आधीचा क्रियापद दरवाजा** जो एजंट धोकादायक टप्पा पार करण्याआधी चालतो, ज्यामुळे खर्च, अपरिवर्तनीयता, आणि विलंब नियंत्रणात राहतात.
2. **धोका स्तरीकरण**, ज्यामुळे कमी धोका असलेले क्रियाकलाप आपोआप चालतात, मध्यम धोका असलेल्या क्रियाकलापांसाठी बॅच-अप्रूव्हल होते, आणि केवळ उच्च धोका असलेल्या क्रियाकलापांना माणसाची अडथळा येतो.
3. एक **ऑडिट लॉग तसेच पुनरावलोकन लूप**, ज्यामुळे प्रत्येक दरवाज्याचा निर्णय JSONL स्वरूपात नोंदवला जातो, आणि नाकारले जाणे एजंटला एक संरचित कारणासह पुन्हा विचारते, फक्त `Revising...` छापण्याऐवजी.

हा नोटबुक `06-system-message-framework.ipynb` प्रमाणेच मूळ घटकांवर प्रत्येक गोष्ट तयार करतो. तो `DEMO_MODE = True` मध्ये अखंड चालतो (काही इंटरऐक्टिव्ह इनपुटची गरज नाही) किंवा `DEMO_MODE = False` असताना खरे `input()` प्रॉम्प्ट्ससह चालतो. लक्षात घ्या: DEMO_MODE मध्ये तिसऱ्या उद्दिष्टाचा रिट्री स्क्रिप्टेड आहे त्यामुळे लूपची यंत्रणा अखंड पाहायला मिळते. खरे पुनरावलोकन-चालित पुनर्गटनेकरिता `DEMO_MODE = False` आणि ऑपरेटर हवा.

**विषयाच्या बाहेर (इतर धड्यात हाताळलेले):** प्रमाणीकरण आणि प्रवेश नियंत्रण (धडा 06 README धोक्याचा भाग 2), साधन कॉल मिडलवेअर (धडा 14 MAF डीप डायव्ह), बहु-एजंट वादविवाद नमुने.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Pattern 1: क्रियापूर्व गेट

README च्या HITL स्निपेटमध्ये एजंटला प्रथम कॉल केला जातो, नंतर वापरकर्त्यास आउटपुट मंजूर करण्यास विचारले जाते. ते म्हणजे **क्रियेनंतरचे** प्रवाह. एजंट आधीच कार्यान्वित झाला आहे, त्यामुळे LLM कॉल खर्च आधीच भरलेला आहे, आणि कोणताही साइड इफेक्ट (इमेल पाठवलेला, डेटाबेस रो लिहिलेला, पोस्ट केलेला टिप्पणी) आधीच घडलेला आहे.

**क्रिया पूर्व** प्रवाहात गेट त्या धोकादायक टप्प्याच्या आधी एजंट चालवण्यापूर्वी ठेवला जातो. एजंट क्रिया सुचवतो, गेट ठरवतो की ती कार्यान्वित करायची की नाही, आणि फक्त मंजुरीवर साइड इफेक्ट घडतो.

| पैलू | क्रियेनंतर मंजुरी (README स्निपेट) | क्रिया पूर्व गेट (हे नोटबुक) |
|---|---|---|
| मंजुरी कधी चालते? | एजंट आउटपुट तयार केल्यानंतर | कोणताही साइड इफेक्ट होण्यापूर्वी |
| नाकारल्यावर LLM खर्च | आधीच भरलेला | केवळ प्रस्तावासाठी भरण्यात आलेला, क्रियेसाठी नाही |
| अपरिवर्तनीय साइड इफेक्ट्स | शक्य (क्रिया आधीच झाली आहे) | प्रतिबंधित |
| तपासणी स्पष्टता | मंजुरी ही एक प्रिंट स्टेटमेंट आहे | मंजुरी ही टाइमस्टँप, क्रिया, कारण यासह JSON नोंद आहे |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## नमुना 2: जोखीम स्तर

प्रत्येक कारवाईस मानवी मंजुरीची गरज नसते. एका सार्वजनिक API कडून वाचन-फक्त शोध ही ग्राहकाला ईमेल पाठवण्यापेक्षा वेगळी जोखीम घेते. दोन्ही एकसारख्या प्रमाणे वागवणे ऑपरेटरचे लक्ष वाया घालवते आणि एजंट हळू करते.

एक सोपे 3-स्तरीय मॉडेल:

| स्तर | उदाहरणे | मंजुरी प्रक्रिया |
|---|---|---|
| `कमी` (वाचन-केवळ) | ज्ञान तक्ता शोधणे, फ्लाइट पर्याय शोधणे, सार्वजनिक वेब पृष्ठ आणणे | स्वयंचलित कार्यान्वयन, लेखापरीक्षणासाठी नोंदवलेले |
| `मध्यम` (सस्ते बदल) | निकाल कॅश करणे, काउंटर वाढवणे, सूचना अनुसूची करणे | स्वयंचलित कार्यान्वयन, पण दररोज पुनरावलोकनासाठी बॅच केलेले |
| `उच्च` (बाह्य-सामना करणारे किंवा अपरिवर्तनीय) | ईमेल पाठवणे, कार्ड आकारणे, सार्वजनिक चॅनेलवर पोस्ट करणे | मानवी मंजुरीवर अडवलेले |

ही एक स्तरवारी आहे. उत्पादन प्रणालींमध्ये बहुदा अधिक सूक्ष्म स्तर वापरले जातात (उदा. AWS IAM परवानगी स्तर, भूमिका-आधारित प्रवेश स्तर). खालील 3-स्तरीय आवृत्ती वाचन-केवळ आणि साइड-इफेक्टिंग क्रिया एकत्र करणाऱ्या एजंटसाठी सर्वात लहान उपयुक्त आवृत्ती आहे.

खालील वर्गीकारक कीवर्ड उसनियन वापरतो जेणेकरून डेमो निश्चित आणि स्वस्त राहील. उत्पादन प्रणालीत तुम्ही याला शिकलेल्या वर्गीकारक किंवा धोरण इंजिनने बदलाल.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## पॅटर्न ३: ऑडिट लॉग आणि पुनरावलोकन लूप

`print("Response approved.")` हा एक ऑडिट लॉग नाही. विश्वासासाठी, प्रत्येक गेट निर्णय संरचित इव्हेंट म्हणून नोंदवला पाहिजे ज्याला तुम्ही नंतर क्वेरी करू शकता, पुन्हा प्ले करू शकता किंवा एखाद्या अपघात पुनरावलोकनास संलग्न करू शकता.

दोन भाग:

1. **फक्त ऍपेंड करणारा JSONL.** प्रत्येक निर्णयासाठी एक ओळ, ज्यात timestamp, क्रिया, टियर, निर्णय, कारण असते. ग्रेप करणे सोपे, नंतर वास्तविक लॉग स्टोअरमध्ये पाठवणे सोपे.
2. **नकारावर पुनरावलोकन लूप.** जेव्हा गेट `deny` परत करते, तेव्हा एजंट स्वतःला नाकारण्याच्या कारणासह पुन्हा प्रॉम्प्ट करतो, जेणेकरून पुढील प्रस्ताव समस्येपासून वाचू शकेल.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## अतिरिक्त साधने

अनेक इतर सार्वजनिक प्रकल्प हे HITL पद्धतींचे विविध प्रकार अंमलात आणतात. आपल्या स्टॅकसाठी कोणता उपाय योग्य आहे ते शोधण्यासाठी पद्धतींचा सामना करा:

- **LangChain** मानव-इन-द-लूप साधन रॅपर्‌स ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): अशी साधने जी मानवी इनपुटसाठी कार्यवाही थांबवतात.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ याने याचे पुनर्रचना केले): एक एजंट भूमिका वापरते जी बहु-एजंट संभाषणांमध्ये माणूस दर्शवते.
- **Semantic Kernel** फंक्शन फिल्टर्स ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): प्रत्येक फंक्शन कॉलच्या भोवती चालणारे मध्यवर्ती स्तर, गेटिंग लॉजिकसाठी योग्य.

प्रत्येक प्रकल्प तीन उप-पॅटर्न्स वेगळ्या प्रकारे हाताळतो: LangChain त्यांना साधनांप्रमाणे रॅप करतो, AutoGen एजंट भूमिका वापरते, Semantic Kernel मध्यवर्ती स्तर फिल्टर्स वापरतो. स्वतःच्या एजंटसाठी डिझाइन निवडण्यापूर्वी एक किंवा दोन अंमलबजावणी पूर्णपणे वाचा.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
